# Qwen entity-matching training

Executable fit preparation, pinned model loading, LoRA training, fit-only hard-negative refresh, and resumable scoring are implemented below. No pretrained model has been downloaded or trained. The integration notebook exercises a tiny random Qwen3 model and synthetic tokenizer only.

Every retained candidate receives an independent yes/no logit difference. A complete reference group contributes one mean pair loss per optimizer step. Model selection uses early_stop; final thresholds use threshold_tune; holdout remains untouched until the policy is locked.

In [ ]:
from pathlib import Path
import json

workspace = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "entity_resolution_research").is_dir()
    and (p / "6ab10eb3b23ba_student_resource").is_dir()
)
PROJECT_DIR = workspace / "anshp"
DATA_DIR = workspace / "6ab10eb3b23ba_student_resource" / "student_resource" / "dataset"
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
REPORT_DIR = PROJECT_DIR / "reports"
OUTPUT_DIR = PROJECT_DIR / "output"
CONFIG = json.loads((PROJECT_DIR / "config.json").read_text(encoding="utf-8"))
assert CONFIG["model"]["id"] == "Qwen/Qwen3-Reranker-0.6B"
print("Project:", PROJECT_DIR)
print("Status: executable Qwen pilot; pretrained model not run.")

## Supervision and split contract

Keep each reference and all its pairs in exactly one group: `fit`, `early_stop`, `threshold_tune`, or `holdout`. Verify external-positive ownership; if an external positive belongs to multiple references, group the connected references together. Audit duplicates across references.

Label retrieved fit candidates using the complete truth set. An explicit empty truth set means singleton; missing truth is an error. Keep all known positives for selected fit references, using supplemental-positive provenance when retrieval missed them, and start with eight hard and two easier negatives per reference. Supplements apply only to fit training and never to evaluation candidates. Require audited external ownership; exclude every candidate owned by a non-fit reference from training and mining, including negative pairs. Refresh hard negatives once using scores on fit candidates only.

For each pair, compute **z = logit(yes) − logit(no)** at the final valid prompt token and optimize **BCEWithLogits(z, label)** independently. A reference may have zero, one, or several matches. There is no softmax across candidates and no full-vocabulary language-model loss. Start with a mean of pair losses within each reference, then average references; reference-aware batching must preserve this weighting across shards.

Training sampling changes class frequencies. Sigmoid outputs are not automatically calibrated deployment probabilities. Tune the acceptance threshold on all naturally retrieved threshold-tune candidates, counting singleton references and unretrieved truth links.

In [ ]:
import hashlib
import math
import random
import re
from collections.abc import Mapping

MODEL_ID = "Qwen/Qwen3-Reranker-0.6B"
FIELD_NAMES = ("business_name", "business_address", "country")
FIELD_LABELS = ("Name", "Address", "Country")
ER_INSTRUCTION = (
    "Decide whether Query and Document identify the same business entity. "
    "Use names, addresses, and countries together. Shared brand or similar "
    "wording alone is insufficient. Missing fields are unknown evidence. "
    "Answer yes for the same entity and no otherwise."
)
QWEN_PREFIX = (
    '<|im_start|>system\nJudge whether the Document meets the requirements based '
    'on the Query and the Instruct provided. Note that the answer can only be '
    '"yes" or "no".<|im_end|>\n<|im_start|>user\n'
)
QWEN_SUFFIX = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"


def field_text(value):
    """Preserve native scripts and string numbers; read source fields as strings."""
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return "[missing]"
    if not isinstance(value, str):
        raise TypeError("Record fields must be strings or explicit missing values.")
    return re.sub(r"\s+", " ", value).strip().replace("<|", "< |") or "[missing]"


def serialize_business_record(record):
    if not isinstance(record, Mapping):
        raise TypeError("A record must be a field mapping.")
    return "\n".join(
        f"{label}: {field_text(record.get(field))}"
        for field, label in zip(FIELD_NAMES, FIELD_LABELS)
    )


def serialize_pair(reference, candidate, instruction=ER_INSTRUCTION):
    if not isinstance(instruction, str) or not instruction.strip():
        raise ValueError("A nonempty business-matching instruction is required.")
    body = (
        f"<Instruct>: {instruction}\n"
        f"<Query>: {serialize_business_record(reference)}\n"
        f"<Document>: {serialize_business_record(candidate)}"
    )
    return QWEN_PREFIX + body + QWEN_SUFFIX


def probability_from_log_odds(log_odds):
    z = float(log_odds)
    if not math.isfinite(z):
        raise ValueError("Log odds must be finite.")
    if z >= 0:
        return 1.0 / (1.0 + math.exp(-z))
    ez = math.exp(z)
    return ez / (1.0 + ez)


def binary_cross_entropy_from_log_odds(log_odds, label):
    """Stable scalar reference equation; torch training uses the same objective."""
    if label not in (0, 1):
        raise ValueError("Binary label must be zero or one.")
    z = float(log_odds)
    if not math.isfinite(z):
        raise ValueError("Log odds must be finite.")
    return max(z, 0.0) - label * z + math.log1p(math.exp(-abs(z)))


## Field-aware token budget

The initial **256 tokens cover the entire pair**, including prefix, instruction, labels, and suffix. Compare 256 with 384/512 during the pilot; 256 is a throughput hypothesis, not the model context limit.

Tokenize field spans separately and reserve the immutable framing first. Allocate remaining capacity fairly across both records, weighting name/address above country. Head-and-tail retention can preserve evidence at both address ends, but not every number. Save per-field truncation diagnostics and audit them by country. Impossible budgets raise errors; the assistant suffix is never truncated.

Model text contains original retained fields with whitespace normalized, explicit missing markers, and no IDs, labels, or retrieval ranks. Source ingestion must use string dtypes to retain leading zeros. Branch-versus-company identity follows the audited truth convention: a different address alone is not a hard-coded nonmatch.

In [ ]:
def _allocate_field_tokens(lengths, budget, weights):
    if len(lengths) != len(weights) or budget < 0:
        raise ValueError("Invalid field budget.")
    allocated = [min(length, 4) for length in lengths]
    if sum(allocated) > budget:
        raise ValueError("Prompt budget cannot preserve all field minima.")
    remaining = budget - sum(allocated)
    schedule = [i for level in range(max(weights))
                for i, weight in enumerate(weights) if weight > level]
    while remaining and any(n < length for n, length in zip(allocated, lengths)):
        for i in schedule:
            if remaining == 0:
                break
            if allocated[i] < lengths[i]:
                allocated[i] += 1
                remaining -= 1
    return allocated


def _head_tail_tokens(tokens, limit, gap_tokens):
    if len(tokens) <= limit:
        return tokens
    if limit <= len(gap_tokens) + 1:
        return tokens[:limit]
    keep = limit - len(gap_tokens)
    head = (keep + 1) // 2
    tail = keep - head
    return tokens[:head] + gap_tokens + (tokens[-tail:] if tail else [])


def encode_pair_with_budget(tokenizer, reference, candidate, max_tokens=256,
                            instruction=ER_INSTRUCTION):
    """Unpadded prompt IDs and diagnostics. No download, model call, or labels."""
    if not isinstance(max_tokens, int) or max_tokens <= 0:
        raise ValueError("max_tokens must be a positive integer.")
    if not isinstance(instruction, str) or not instruction.strip():
        raise ValueError("A nonempty business-matching instruction is required.")
    if not isinstance(reference, Mapping) or not isinstance(candidate, Mapping):
        raise TypeError("Records must be field mappings.")
    enc = lambda text: list(tokenizer.encode(text, add_special_tokens=False))
    prefix = enc(QWEN_PREFIX + f"<Instruct>: {instruction}")
    suffix = enc(QWEN_SUFFIX)
    label_parts, field_parts, names = [], [], []
    for role, record in (("Query", reference), ("Document", candidate)):
        for index, (field, label) in enumerate(zip(FIELD_NAMES, FIELD_LABELS)):
            label_text = f"\n<{role}>: {label}: " if index == 0 else f"\n{label}: "
            label_parts.append(enc(label_text))
            field_parts.append(enc(field_text(record.get(field))))
            names.append(f"{role}.{field}")
    fixed_count = len(prefix) + len(suffix) + sum(map(len, label_parts))
    lengths = list(map(len, field_parts))
    limits = _allocate_field_tokens(
        lengths, max_tokens - fixed_count, weights=(3, 5, 1, 3, 5, 1)
    )
    gap = enc(" … ")
    ids = list(prefix)
    diagnostics = {}
    for name, label, value, limit in zip(names, label_parts, field_parts, limits):
        retained = _head_tail_tokens(value, limit, gap)
        ids.extend(label)
        ids.extend(retained)
        diagnostics[name] = {
            "original_tokens": len(value),
            "retained_tokens_including_gap": len(retained),
            "truncated": len(value) > limit,
        }
    ids.extend(suffix)
    assert len(ids) <= max_tokens
    assert ids[-len(suffix):] == suffix
    return {"input_ids": ids, "attention_mask": [1] * len(ids),
            "field_diagnostics": diagnostics}


def yes_no_token_ids(tokenizer):
    ids = {}
    for word in ("yes", "no"):
        encoded = list(tokenizer.encode(word, add_special_tokens=False))
        token_id = tokenizer.convert_tokens_to_ids(word)
        if len(encoded) != 1 or encoded[0] != token_id:
            raise ValueError(f"Expected one exact token for {word!r}.")
        ids[word] = token_id
    if ids["yes"] == ids["no"]:
        raise ValueError("yes/no token IDs must differ.")
    return ids["yes"], ids["no"]


## Fit-only sampling and external-record ownership

Pass unique candidate IDs ordered from harder to easier. Initially use retrieval order; before model-based mining, apply the same external-ownership eligibility filter, then rank **eligible fit negatives only** by Qwen score. Keep an easier-negative mixture and all known positives for selected fit references. Log checkpoint, candidate version, ownership audit, seed, class counts, and inclusion policy.

The helper requires an explicit audited external_owner mapping: each external ID maps to its owning reference, or to None only if the complete truth audit establishes that it is unmatched. Missing entries and owners with unknown splits are errors. Gold-positive ownership must agree with the fit query. Candidates owned by early_stop, threshold_tune, or holdout references are excluded even as negatives; this prevents their text from entering fitting or hard-negative mining. Other fit references' positives may serve as negative pairs under the audited single-owner convention.

By default, fit positives absent from retrieval are retained as supplemental_positive examples. This never changes measured retrieval recall or evaluation candidates. Singleton references contribute eligible negatives. Early-stop, threshold-tune, and holdout keep natural candidates, without sampling or injected positives. Entire candidate groups and a suitable batch denominator are required for the per-reference loss policy.


In [ ]:
def make_fit_examples(reference_id, ranked_candidate_ids, truth_by_reference,
                      split_by_reference, external_owner, hard_negatives=8,
                      easy_negatives=2, seed=2026,
                      include_supplemental_positives=True):
    """Fit pairs with explicit ownership checks; evaluation-owned records excluded."""
    if split_by_reference.get(reference_id) != "fit":
        raise ValueError("Training/mining is restricted to fit references.")
    if reference_id not in truth_by_reference:
        raise KeyError("Missing truth entry cannot be interpreted as a singleton.")
    if hard_negatives < 0 or easy_negatives < 0:
        raise ValueError("Negative sample counts must be nonnegative.")
    if not isinstance(external_owner, Mapping):
        raise TypeError("An audited external-ID ownership mapping is required.")
    candidates = list(ranked_candidate_ids)
    if len(set(candidates)) != len(candidates):
        raise ValueError("Candidates must be unique per reference.")
    positives = set(truth_by_reference[reference_id])
    allowed_splits = {"fit", "early_stop", "threshold_tune", "holdout"}

    def audited_owner(candidate_id):
        if candidate_id not in external_owner:
            raise KeyError(f"External ownership was not audited for {candidate_id!r}.")
        owner = external_owner[candidate_id]
        if owner is not None and split_by_reference.get(owner) not in allowed_splits:
            raise ValueError(f"Unknown reference split for owner {owner!r}.")
        return owner

    # Check all gold positives, including those absent from retrieval.
    for candidate_id in positives:
        if audited_owner(candidate_id) != reference_id:
            raise ValueError("A gold positive must be owned by the fit reference.")
    eligible = []
    for candidate_id in candidates:
        owner = audited_owner(candidate_id)
        if owner == reference_id and candidate_id not in positives:
            raise ValueError("Ownership and complete query truth disagree.")
        if owner is None or split_by_reference[owner] == "fit":
            eligible.append(candidate_id)
        # Exclude external positives owned by every non-fit reference group,
        # even when their pair label for this query would be negative.

    positive_ids = [candidate for candidate in eligible if candidate in positives]
    supplemental_ids = (
        sorted(positives.difference(candidates), key=str)
        if include_supplemental_positives else []
    )
    negative_ids = [candidate for candidate in eligible if candidate not in positives]
    hard = negative_ids[:hard_negatives]
    remainder = negative_ids[hard_negatives:]
    stable_seed = int.from_bytes(
        hashlib.sha256(f"{seed}:{reference_id}".encode("utf-8")).digest()[:8], "big"
    )
    easy = random.Random(stable_seed).sample(remainder, min(easy_negatives, len(remainder)))
    sampled = (
        [(candidate, 1, "retrieved_positive") for candidate in positive_ids]
        + [(candidate, 1, "supplemental_positive") for candidate in supplemental_ids]
        + [(candidate, 0, "hard_negative") for candidate in hard]
        + [(candidate, 0, "easy_negative") for candidate in easy]
    )
    count = len(sampled)
    return [
        {"reference_id": reference_id, "candidate_id": candidate,
         "label": label, "sampling_reason": reason, "reference_weight": 1.0 / count}
        for candidate, label, reason in sampled
    ]


## Reusable tensor scorer

The following definitions import PyTorch only when called. Inference can load cells tagged `er-definitions` without running bootstrap or training. Supply prompt-only inputs to `AutoModelForCausalLM`; **never append the target yes/no label**. Reuse the pinned tokenizer and encoding contract.

Use left padding, attention masks, and length buckets. The scorer explicitly locates the final valid token, also supporting right padding. Full sequence/vocabulary logits consume substantial memory; benchmark a runtime-supported last-token path later and verify parity. Training needs gradients; inference uses `model.eval()` and `torch.inference_mode()`. Raw log-odds are the primary score.

In [ ]:
def qwen_pair_log_odds(logits, attention_mask, yes_id, no_id):
    """Extract independent yes/no log odds at each row's last valid token."""
    import torch

    if logits.ndim != 3 or attention_mask.ndim != 2:
        raise ValueError("Expected [batch, sequence, vocabulary] logits and 2-D mask.")
    if tuple(logits.shape[:2]) != tuple(attention_mask.shape):
        raise ValueError("Mask and logits must describe the same prompts.")
    if yes_id == no_id or min(yes_id, no_id) < 0 or max(yes_id, no_id) >= logits.shape[-1]:
        raise ValueError("Invalid yes/no vocabulary IDs.")
    mask = attention_mask.to(device=logits.device)
    if not bool(((mask == 0) | (mask == 1)).all()):
        raise ValueError("Attention mask must contain only zero and one.")
    valid = mask.bool()
    if not bool(valid.any(dim=1).all()):
        raise ValueError("An all-padding prompt cannot be scored.")
    positions = torch.arange(logits.shape[1], device=logits.device).expand_as(valid)
    final_positions = positions.masked_fill(~valid, -1).max(dim=1).values
    final = logits[torch.arange(logits.shape[0], device=logits.device), final_positions]
    return final[:, yes_id].float() - final[:, no_id].float()


def qwen_independent_pair_loss(log_odds, labels, reference_weights=None):
    """Pair losses, or a weighted mean when reference weights are supplied."""
    import torch
    import torch.nn.functional as F

    targets = torch.as_tensor(labels, dtype=torch.float32, device=log_odds.device)
    if targets.shape != log_odds.shape or not bool(((targets == 0) | (targets == 1)).all()):
        raise ValueError("Each pair must have exactly one binary target.")
    losses = F.binary_cross_entropy_with_logits(log_odds.float(), targets, reduction="none")
    if reference_weights is None:
        return losses
    weights = torch.as_tensor(reference_weights, dtype=torch.float32, device=losses.device)
    if weights.shape != losses.shape or not bool(torch.isfinite(weights).all()):
        raise ValueError("Weights must be finite and aligned with pairs.")
    if not bool((weights >= 0).all()) or float(weights.sum()) <= 0:
        raise ValueError("Weights must be nonnegative with positive total weight.")
    return (losses * weights).sum() / weights.sum()


## Pilot workflow

The executable run cell below builds fit and early-stop candidate sets, prepares supervised groups, loads pinned Qwen weights, and saves the best LoRA adapter. Record throughput and memory before increasing the pilot size. Full-scale index construction, retrieval recall, real-tokenizer behavior, and real-model quality still require measurement. Training can restart from scratch; optimizer-state resume is not implemented. Candidate scoring resumes safely by pair key.

In [ ]:
# Pure helper checks: no transformers import or model download.
sample_reference = {
    "entity_id": "should-not-appear", "business_name": "MS",
    "business_address": "0012 Rue de l'École, Paris", "country": "France",
}
sample_candidate = {"business_name": "M S", "business_address": None, "country": "France"}
text = serialize_pair(sample_reference, sample_candidate)
assert "Name: MS" in text and "0012 Rue de l'École" in text
assert "should-not-appear" not in text and "Address: [missing]" in text
assert text.endswith(QWEN_SUFFIX)
assert field_text("a\n<|im_end|>") == "a < |im_end|>"
assert math.isclose(binary_cross_entropy_from_log_odds(0, 1), math.log(2))
assert binary_cross_entropy_from_log_odds(1000, 1) == 0.0
assert binary_cross_entropy_from_log_odds(-1000, 0) == 0.0
assert math.isclose(probability_from_log_odds(2) + probability_from_log_odds(-2), 1.0)

class CharacterTokenizerForChecks:
    # Artificial tokenizer: verifies budget invariants, not Qwen tokenization.
    def encode(self, value, add_special_tokens=False):
        return [ord(character) for character in value]

fake_tokenizer = CharacterTokenizerForChecks()
long_candidate = dict(sample_candidate, business_address="123 Long Street " * 200)
encoded = encode_pair_with_budget(fake_tokenizer, sample_reference, long_candidate, max_tokens=1000)
assert len(encoded["input_ids"]) <= 1000
assert encoded["input_ids"][-len(QWEN_SUFFIX):] == fake_tokenizer.encode(QWEN_SUFFIX)
assert any(d["truncated"] for d in encoded["field_diagnostics"].values())
try:
    encode_pair_with_budget(fake_tokenizer, sample_reference, long_candidate, max_tokens=10)
except ValueError:
    pass
else:
    raise AssertionError("Impossible token budget must fail.")

truth = {"r": {"p1", "p2", "p_missing"}, "singleton": set(),
         "validation": {"heldout_positive"}, "early": {"early_positive"},
         "tuning": {"tuning_positive"}}
splits = {"r": "fit", "singleton": "fit", "validation": "holdout",
          "early": "early_stop", "tuning": "threshold_tune"}
owners = {"p1": "r", "p2": "r", "p_missing": "r",
          "n1": None, "n2": None, "n3": None,
          "heldout_positive": "validation", "early_positive": "early",
          "tuning_positive": "tuning"}
ranked_ids = ["heldout_positive", "early_positive", "tuning_positive",
              "n1", "p1", "n2", "p2", "n3"]
examples = make_fit_examples("r", ranked_ids, truth, splits, owners,
                             hard_negatives=1, easy_negatives=1)
assert {e["candidate_id"] for e in examples if e["label"]} == {"p1", "p2", "p_missing"}
assert not {"heldout_positive", "early_positive", "tuning_positive"}.intersection(
    e["candidate_id"] for e in examples
)
assert next(e for e in examples if e["candidate_id"] == "p_missing")[
    "sampling_reason"] == "supplemental_positive"
assert math.isclose(sum(e["reference_weight"] for e in examples), 1.0)
assert examples == make_fit_examples("r", ranked_ids, truth, splits, owners,
                                     hard_negatives=1, easy_negatives=1)
singleton_examples = make_fit_examples(
    "singleton", ["heldout_positive", "n1"], truth, splits, owners
)
assert len(singleton_examples) == 1
assert singleton_examples[0]["candidate_id"] == "n1" and singleton_examples[0]["label"] == 0
assert make_fit_examples("singleton", [], truth, splits, owners) == []

def expect_sampling_error(exception_type, *args, **kwargs):
    try:
        make_fit_examples(*args, **kwargs)
    except exception_type:
        return
    raise AssertionError(f"Expected {exception_type.__name__}.")

expect_sampling_error(ValueError, "validation", ["n1"], truth, splits, owners)
expect_sampling_error(KeyError, "r", ["not_audited"], truth, splits, owners)
expect_sampling_error(ValueError, "r", ["unknown_split"], truth, splits,
                      dict(owners, unknown_split="missing_reference"))
expect_sampling_error(ValueError, "r", ["p1"], truth, splits,
                      dict(owners, p1="validation"))
expect_sampling_error(ValueError, "singleton", ["p1"], truth, splits,
                      dict(owners, p1="singleton"))
print("Pure helper and ownership-isolation checks passed; see notebook 06 for tensor/training integration.")


## Executable training and scoring

LoRA loading requires pinned commits. Fit preparation filters external ownership in SQL and supplements missing gold positives only during fitting. The loop normalizes across complete reference groups, refreshes fit-only hard negatives once, evaluates complete early-stop candidates, and saves the best adapter. Scoring resumes by pair key and binds output to model, prompt, and candidate identities.

In [ ]:
from pathlib import Path
import json
_ROOT = Path(__file__).resolve().parent if "__file__" in globals() else next(
    p / "anshp" for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "anshp" / "config.json").is_file())
_A = {"__file__": str(_ROOT / "00_setup_and_data_audit.ipynb")}
for _c in json.loads((_ROOT / "00_setup_and_data_audit.ipynb").read_text(encoding="utf-8"))["cells"]:
    if _c["cell_type"] == "code" and "er-definitions" in _c.get("metadata", {}).get("tags", []):
        exec("".join(_c["source"]), _A)

import itertools
import sqlite3
import time
from functools import lru_cache
_B, _ = _A["load_helpers"](_ROOT / "02_blocking.ipynb")

def pin_qwen_revisions(config_path):
    """Explicit network operation. Persist immutable commits before any model load."""
    from huggingface_hub import HfApi
    config_path=Path(config_path)
    config=json.loads(config_path.read_text(encoding="utf-8"))
    for id_key,rev_key in (("id","revision"),("tokenizer_id","tokenizer_revision")):
        current=config["model"].get(rev_key) or "main"
        config["model"][rev_key]=HfApi().model_info(config["model"][id_key],revision=current).sha
    _A["atomic_json"](config_path,config)
    return config

def load_qwen(config, train=False, adapter_dir=None, local_files_only=False):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import LoraConfig, TaskType, get_peft_model, PeftModel
    spec=config["model"]
    if config.get("synthetic_fixture"):
        raise ValueError("Fixture artifacts cannot load a production model")
    for key in ("revision","tokenizer_revision"):
        if not re.fullmatch(r"[0-9a-f]{40}",spec.get(key) or ""):
            raise ValueError("Pin immutable model and tokenizer revisions first")
    cache_dir=_ROOT/"artifacts/huggingface_cache"
    cache_dir.mkdir(parents=True,exist_ok=True)
    model_source,tokenizer_source=spec["id"],spec["tokenizer_id"]
    if local_files_only:
        from huggingface_hub import snapshot_download
        model_source=snapshot_download(spec["id"],revision=spec["revision"],cache_dir=str(cache_dir),local_files_only=True)
        tokenizer_source=snapshot_download(spec["tokenizer_id"],revision=spec["tokenizer_revision"],cache_dir=str(cache_dir),local_files_only=True)
    tokenizer=AutoTokenizer.from_pretrained(tokenizer_source,cache_dir=str(cache_dir),
        revision=spec["tokenizer_revision"],trust_remote_code=False,padding_side="left",local_files_only=local_files_only)
    if tokenizer.pad_token_id is None:tokenizer.pad_token=tokenizer.eos_token
    if tokenizer.pad_token_id is None:raise ValueError("Tokenizer has no padding token")
    yes_id,no_id=yes_no_token_ids(tokenizer)
    device="cuda" if torch.cuda.is_available() else "cpu"
    dtype=(torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16) \
          if device=="cuda" else torch.float32
    model=AutoModelForCausalLM.from_pretrained(model_source,revision=spec["revision"],cache_dir=str(cache_dir),
        torch_dtype=dtype,trust_remote_code=False,use_safetensors=True,local_files_only=local_files_only)
    model.to(device)
    model.config.use_cache=False
    if adapter_dir is not None:
        model=PeftModel.from_pretrained(model,str(adapter_dir),is_trainable=train)
    elif train:
        lora=config["training"]["lora"]
        model=get_peft_model(model,LoraConfig(task_type=TaskType.CAUSAL_LM,
            r=lora["r"],lora_alpha=lora["alpha"],lora_dropout=lora["dropout"],
            target_modules=lora["target_modules"],bias="none"))
    if train and config["training"]["gradient_checkpointing"]:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant":False})
        model.enable_input_require_grads()
    model.train(train)
    return model,tokenizer,(yes_id,no_id)

def pair_batch(tokenizer, pairs, device, max_tokens):
    encoded=[encode_pair_with_budget(tokenizer,q,d,max_tokens) for q,d in pairs]
    diagnostics=[e.pop("field_diagnostics") for e in encoded]
    batch=tokenizer.pad(encoded,padding=True,return_tensors="pt")
    batch={k:v.to(device) for k,v in batch.items()}
    if not bool((batch["attention_mask"][:,-1]==1).all()):
        raise ValueError("Last-token-only scoring requires left padding")
    return batch,diagnostics

def forward_pair_scores(model,batch,yes_id,no_id):
    # Qwen3 supports logits_to_keep; avoid full [batch,sequence,vocabulary] allocation.
    output=model(**batch,logits_to_keep=1,use_cache=False)
    if output.logits.shape[1]!=1:
        raise ValueError("Model did not honor last-token logits")
    return output.logits[:,0,yes_id].float()-output.logits[:,0,no_id].float()

def record_lookup(con):
    @lru_cache(maxsize=10000)
    def get(entity_id):
        row=con.execute("SELECT business_name,business_address,country FROM records WHERE entity_id=?",
                        (entity_id,)).fetchone()
        if row is None:raise ValueError(f"Missing normalized record: {entity_id}")
        return dict(zip(FIELD_NAMES,row))
    return get

def prepare_fit_data(candidate_dir,output_dir,config):
    """Join ownership in SQL; retain all known positives and only fit-eligible negatives."""
    candidate_dir,out=Path(candidate_dir),Path(output_dir);out.mkdir(parents=True,exist_ok=True)
    cm=_A["require_manifest"](candidate_dir/"manifest.json","candidates")
    if cm["dataset"]!="train":raise ValueError("Training requires train candidates")
    audit_dir=Path(cm["audit_dir"]);audit=_A["require_manifest"](audit_dir/"manifest.json","audit")
    identity={"candidates":cm["fingerprint"],"training":config["training"],"seed":config["seed"],
              "helpers":_A["load_helpers"](_ROOT/"03_qwen_training.ipynb")[1]}
    fp=_A["stable_hash"](identity)
    if (out/"manifest.json").exists():
        old=_A["require_manifest"](out/"manifest.json","fit_data")
        if old["fingerprint"]==fp:return old
    con=_A["connect_database"](audit_dir/audit["database"],read_only=True)
    temporary=out/f"fit-{_A['uuid'].uuid4().hex}.tmp.sqlite"
    db=sqlite3.connect(temporary)
    try:
        for table in ("query_scope","candidates"):
            p=next(candidate_dir/f["path"] for f in cm["files"]
                   if f["path"].startswith(table) and f["path"].endswith(".parquet"))
            con.execute(f"CREATE TEMP VIEW {table} AS SELECT * FROM read_parquet({_A['sql_literal'](p)})")
        # A whole-training candidate set is allowed; selecting fit never exposes non-fit labels.
        con.execute("""CREATE TEMP TABLE selected AS SELECT q.reference_id FROM query_scope q
            JOIN splits s USING(reference_id) WHERE s.split='fit'
            ORDER BY md5(?||q.reference_id),q.reference_id LIMIT ?""",
            [str(config["seed"]),config["training"]["pilot_reference_count"]])
        db.execute("CREATE TABLE scope(reference_id TEXT PRIMARY KEY)")
        db.executemany("INSERT INTO scope VALUES (?)",con.execute("SELECT reference_id FROM selected").fetchall())
        db.execute("""CREATE TABLE eligible(reference_id TEXT,candidate_id TEXT,label INTEGER,
            rank INTEGER,retrieved INTEGER,PRIMARY KEY(reference_id,candidate_id))""")
        db.execute("""CREATE TABLE examples(reference_id TEXT,candidate_id TEXT,label INTEGER,
            sampling_reason TEXT,reference_weight REAL,PRIMARY KEY(reference_id,candidate_id))""")
        cursor=con.execute("""WITH eligible_candidates AS (
            SELECT c.reference_id,c.candidate_id,(t.candidate_id IS NOT NULL)::INTEGER AS pair_label,
                   c.rank,1 retrieved FROM candidates c JOIN selected s USING(reference_id)
            LEFT JOIN truth t USING(reference_id,candidate_id)
            LEFT JOIN ownership o ON o.candidate_id=c.candidate_id
            WHERE o.owner_reference_id IS NULL OR o.owner_split='fit'),
            supplements AS (
            SELECT t.reference_id,t.candidate_id,1 AS pair_label,2147483647 rank,0 retrieved
            FROM truth t JOIN selected s USING(reference_id)
            WHERE NOT EXISTS (SELECT 1 FROM candidates c WHERE c.reference_id=t.reference_id
                              AND c.candidate_id=t.candidate_id))
            SELECT * FROM eligible_candidates UNION ALL SELECT * FROM supplements""")
        while rows:=cursor.fetchmany(20000):
            db.executemany("INSERT INTO eligible VALUES (?,?,?,?,?)",rows)
        resample_fit_examples(db,config)
        count=db.execute("SELECT count(*) FROM scope").fetchone()[0]
        pairs=db.execute("SELECT count(*) FROM examples").fetchone()[0]
        empty=db.execute("SELECT count(*) FROM scope s WHERE NOT EXISTS (SELECT 1 FROM examples e WHERE e.reference_id=s.reference_id)").fetchone()[0]
        if not pairs:raise ValueError("No eligible supervised pairs")
        db.commit();db.close();db=None
        target=out/f"fit-{fp[:16]}.sqlite";temporary.replace(target)
        m={"stage":"fit_data","complete":True,"fingerprint":fp,"identity":identity,
           "database":target.name,"files":[_A["file_entry"](target,out)],
           "references":count,"references_without_pairs":empty,"pairs":pairs,
           "index_dir":cm["index_dir"],"candidate_dir":str(candidate_dir.resolve())}
        _A["atomic_json"](out/"manifest.json",m)
        return m
    finally:
        con.close()
        if db is not None:db.close()

def resample_fit_examples(db,config,rankings=None):
    db.execute("DELETE FROM examples")
    refs=db.execute("SELECT reference_id FROM scope ORDER BY reference_id").fetchall()
    for (ref,) in refs:
        rows=db.execute("""SELECT candidate_id,label,retrieved FROM eligible
                           WHERE reference_id=? ORDER BY rank,candidate_id""",(ref,)).fetchall()
        positives={cid for cid,label,_ in rows if label}
        candidates=[cid for cid,_,retrieved in rows if retrieved]
        if rankings is not None:
            candidates.sort(key=lambda cid:(-rankings.get((ref,cid),float("-inf")),cid))
        # SQL already excluded all non-fit owners. This local mapping is sufficient
        # for the helper's ownership gate: negatives are audited eligible, not unknown.
        owners={cid:ref if label else None for cid,label,_ in rows}
        examples=make_fit_examples(ref,candidates,{ref:positives},{ref:"fit"},owners,
            config["training"]["hard_negatives_per_reference"],
            config["training"]["easy_negatives_per_reference"],config["seed"])
        db.executemany("INSERT INTO examples VALUES (?,?,?,?,?)",
            [(e["reference_id"],e["candidate_id"],e["label"],e["sampling_reason"],
              e["reference_weight"]) for e in examples])
    db.commit()

def score_candidates(model,tokenizer,yes_no,candidate_dir,output_dir,config,model_identity,
                     batch_size=None):
    """Resume by pair key, binding partial output to model/data/prompt identities."""
    import torch
    candidate_dir,out=Path(candidate_dir),Path(output_dir);out.mkdir(parents=True,exist_ok=True)
    cm=_A["require_manifest"](candidate_dir/"manifest.json","candidates")
    index_dir=Path(cm["index_dir"]);im=_A["require_manifest"](index_dir/"manifest.json","index")
    if im["fingerprint"]!=cm["identity"]["index"]:raise ValueError("Candidate/index mismatch")
    helper_hash=_A["load_helpers"](_ROOT/"03_qwen_training.ipynb")[1]
    identity={"candidates":cm["fingerprint"],"model":model_identity,
              "model_config":config["model"],"helpers":helper_hash}
    fp=_A["stable_hash"](identity)
    if (out/"manifest.json").exists():
        old=_A["require_manifest"](out/"manifest.json","scores")
        if old["fingerprint"]==fp:return old
    target=out/f"scores-{fp[:16]}.sqlite"
    db=sqlite3.connect(target)
    db.execute("""CREATE TABLE IF NOT EXISTS scores(reference_id TEXT,candidate_id TEXT,
                   score REAL NOT NULL,PRIMARY KEY(reference_id,candidate_id))""")
    db.execute("CREATE TABLE IF NOT EXISTS metadata(fingerprint TEXT)")
    old=db.execute("SELECT fingerprint FROM metadata").fetchall()
    if old and old!=[(fp,)]:db.close();raise ValueError("Partial score identity mismatch")
    if not old:db.execute("INSERT INTO metadata VALUES (?)",(fp,));db.commit()
    cc=_B["sqlite_readonly"](candidate_dir/cm["database"])
    rc=_B["sqlite_readonly"](index_dir/im["database"]);get=record_lookup(rc)
    was_training=model.training;model.eval()
    batch_size=batch_size or config["training"]["micro_batch_size"]
    started=time.perf_counter();new_pairs=0;truncated=0;field_count=0
    try:
        cursor=cc.execute("SELECT reference_id,candidate_id FROM candidates ORDER BY reference_id,rank")
        with torch.inference_mode():
            while rows:=cursor.fetchmany(batch_size):
                pending=[(q,c) for q,c in rows if db.execute(
                    "SELECT 1 FROM scores WHERE reference_id=? AND candidate_id=?",(q,c)).fetchone() is None]
                if not pending:continue
                batch,diagnostics=pair_batch(tokenizer,[(get(q),get(c)) for q,c in pending],
                                            next(model.parameters()).device,config["model"]["max_pair_tokens"])
                z=forward_pair_scores(model,batch,*yes_no)
                if not bool(torch.isfinite(z).all()):raise ValueError("Nonfinite model scores")
                db.executemany("INSERT INTO scores VALUES (?,?,?)",
                    [(q,c,float(v)) for (q,c),v in zip(pending,z.cpu().tolist())])
                db.commit();new_pairs+=len(pending)
                field_count+=sum(len(d) for d in diagnostics)
                truncated+=sum(v["truncated"] for d in diagnostics for v in d.values())
        count=db.execute("SELECT count(*) FROM scores").fetchone()[0]
        if count!=cm["pairs"]:raise ValueError("Not every final candidate was scored")
        db.close();db=None
        result={"stage":"scores","complete":True,"fingerprint":fp,"identity":identity,
            "database":target.name,"files":[_A["file_entry"](target,out)],
            "candidate_dir":str(candidate_dir.resolve()),"pairs":count,
            "profile":{"new_pairs":new_pairs,"seconds":time.perf_counter()-started,
                       "truncated_fields":truncated,"fields":field_count,
                       "peak_gpu_allocated_bytes":torch.cuda.max_memory_allocated() if torch.cuda.is_available() else None}}
        _A["atomic_json"](out/"manifest.json",result)
        return result
    finally:
        if db is not None:db.close()
        cc.close();rc.close();model.train(was_training)

def train_qwen(model,tokenizer,yes_no,fit_dir,output_dir,config,early_stop_evaluator):
    """Pilot LoRA loop. Evaluator must return macro_f05 on complete early_stop candidates."""
    import torch
    from transformers import get_linear_schedule_with_warmup, set_seed
    set_seed(config["seed"])
    if not callable(early_stop_evaluator):raise ValueError("An early-stop evaluator is required")
    if not any(p.requires_grad for p in model.parameters()):raise ValueError("No trainable adapter")
    fit_dir,out=Path(fit_dir),Path(output_dir);out.mkdir(parents=True,exist_ok=True)
    fit=_A["require_manifest"](fit_dir/"manifest.json","fit_data")
    index_dir=Path(fit["index_dir"]);im=_A["require_manifest"](index_dir/"manifest.json","index")
    # Copy supervision so negative refresh never mutates the published fit-data artifact.
    source=_B["sqlite_readonly"](fit_dir/fit["database"])
    run_db=out/f"training-{_A['uuid'].uuid4().hex}.sqlite"
    db=sqlite3.connect(run_db);source.backup(db);source.close()
    rc=_B["sqlite_readonly"](index_dir/im["database"]);get=record_lookup(rc)
    refs=[r[0] for r in db.execute("SELECT DISTINCT reference_id FROM examples ORDER BY reference_id")]
    settings=config["training"]
    groups_per_step=settings.get("reference_groups_per_step",8)
    micro=settings["micro_batch_size"]
    epochs=settings["max_epochs"]
    steps=math.ceil(len(refs)/groups_per_step)*epochs
    optimizer=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
        lr=settings["learning_rate"],weight_decay=settings["weight_decay"])
    scheduler=get_linear_schedule_with_warmup(optimizer,int(steps*settings["warmup_ratio"]),steps)
    use_scaler=next(model.parameters()).device.type=="cuda" and next(model.parameters()).dtype==torch.float16
    scaler=torch.amp.GradScaler("cuda",enabled=use_scaler)
    helper_hash=_A["load_helpers"](_ROOT/"03_qwen_training.ipynb")[1]
    best=-1.0;history=[];started=time.perf_counter();pairs_seen=0
    try:
        for epoch in range(epochs):
            if epoch==1 and settings.get("hard_negative_refreshes",0):
                # Score eligible negatives only, then resample with unchanged complete positives.
                # Rankings are bounded by the configured pilot reference count.
                rankings={};model.eval()
                cursor=db.execute("SELECT reference_id,candidate_id FROM eligible WHERE label=0")
                with torch.inference_mode():
                    while rows:=cursor.fetchmany(micro):
                        batch,_=pair_batch(tokenizer,[(get(q),get(c)) for q,c in rows],
                            next(model.parameters()).device,config["model"]["max_pair_tokens"])
                        z=forward_pair_scores(model,batch,*yes_no)
                        if not bool(torch.isfinite(z).all()):raise ValueError("Nonfinite mining scores")
                        rankings.update({pair:float(v) for pair,v in zip(rows,z.cpu().tolist())})
                resample_fit_examples(db,config,rankings)
            random.Random(config["seed"]+epoch).shuffle(refs);model.train()
            running=0.0
            for start in range(0,len(refs),groups_per_step):
                group_ids=refs[start:start+groups_per_step];examples=[]
                for ref in group_ids:
                    examples.extend(db.execute("""SELECT reference_id,candidate_id,label,reference_weight
                        FROM examples WHERE reference_id=? ORDER BY candidate_id""",(ref,)).fetchall())
                random.Random(config["seed"]+epoch+start).shuffle(examples)
                optimizer.zero_grad(set_to_none=True)
                for offset in range(0,len(examples),micro):
                    chunk=examples[offset:offset+micro]
                    batch,_=pair_batch(tokenizer,[(get(q),get(c)) for q,c,_,_ in chunk],
                        next(model.parameters()).device,config["model"]["max_pair_tokens"])
                    z=forward_pair_scores(model,batch,*yes_no)
                    if not bool(torch.isfinite(z).all()):raise ValueError("Nonfinite training logits")
                    losses=qwen_independent_pair_loss(z,[r[2] for r in chunk])
                    weights=torch.tensor([r[3] for r in chunk],device=z.device)
                    # A common denominator across ALL microbatches in this optimizer step.
                    # Each reference contributes its mean pair loss, even when split across batches.
                    loss=(losses*weights).sum()/len(group_ids)
                    scaler.scale(loss).backward();running+=float(loss.detach());pairs_seen+=len(chunk)
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.0,error_if_nonfinite=True)
                scaler.step(optimizer);scaler.update();scheduler.step()
            metrics=early_stop_evaluator(model,epoch)
            value=float(metrics["macro_f05"])
            if not math.isfinite(value) or not 0<=value<=1:raise ValueError("Invalid early-stop score")
            history.append({"epoch":epoch+1,"macro_f05":value,"step_loss_sum":running})
            if value>best:
                best=value
                checkpoint=out/f"best-epoch-{epoch+1}"
                model.save_pretrained(checkpoint,safe_serialization=True)
                tokenizer.save_pretrained(checkpoint/"tokenizer")
                files=[_A["file_entry"](p,out) for p in checkpoint.rglob("*") if p.is_file()]
                manifest={"stage":"model","complete":True,"fingerprint":_A["stable_hash"]({
                    "fit":fit["fingerprint"],"config":config,"helpers":helper_hash,"epoch":epoch+1,
                    "checkpoint_files":files}),
                    "files":files,"adapter":str(checkpoint.relative_to(out)),
                    "config":config,"qwen_helpers_sha256":helper_hash,
                    "normalization_fingerprint":im["identity"]["normalized"],
                    "fit_fingerprint":fit["fingerprint"],"best_early_stop_macro_f05":best,
                    "history":list(history),"environment":_A["environment_snapshot"](),
                    "training_profile":{"seconds":time.perf_counter()-started,"pairs_seen":pairs_seen,
                        "peak_gpu_allocated_bytes":torch.cuda.max_memory_allocated() if torch.cuda.is_available() else None}}
                _A["atomic_json"](out/"manifest.json",manifest)
            _A["atomic_json"](out/"training_history.json",history)
        return _A["require_manifest"](out/"manifest.json","model")
    finally:db.close();rc.close()


In [ ]:
from pathlib import Path
import json
PROJECT_DIR = next(p / "anshp" for p in (Path.cwd(), *Path.cwd().parents)
                   if (p / "anshp" / "config.json").is_file())
DATA_DIR = PROJECT_DIR.parent / "6ab10eb3b23ba_student_resource" / "student_resource" / "dataset"
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
CONFIG = json.loads((PROJECT_DIR / "config.json").read_text(encoding="utf-8"))

PIN_REVISIONS = False
RUN_TRAINING = False
if PIN_REVISIONS:
    CONFIG = pin_qwen_revisions(PROJECT_DIR/"config.json")
if RUN_TRAINING:
    from transformers import set_seed
    set_seed(CONFIG["seed"])
    cap = CONFIG["blocking"]["selected_candidate_cap"] or 40  # Explicit pilot hypothesis.
    for split,limit in (("fit",CONFIG["training"]["pilot_reference_count"]),("early_stop",1000)):
        _B["build_candidates"](ARTIFACT_DIR/"index_train",ARTIFACT_DIR/"audit",
            ARTIFACT_DIR/("candidates_"+split),cap=cap,split=split,reference_limit=limit,
            channel_limit=CONFIG["blocking"]["channel_limit"])
    prepare_fit_data(ARTIFACT_DIR/"candidates_fit",ARTIFACT_DIR/"fit",CONFIG)
    model,tokenizer,yes_no = load_qwen(CONFIG,train=True)
    evaluation,_ = _A["load_helpers"](PROJECT_DIR/"04_validation_and_thresholds.ipynb")
    run_id = _A["uuid"].uuid4().hex
    def early_stop_evaluator(current_model,epoch):
        score_dir=ARTIFACT_DIR/"training_scores"/f"{run_id}-epoch-{epoch}"
        score_candidates(current_model,tokenizer,yes_no,ARTIFACT_DIR/"candidates_early_stop",
                         score_dir,CONFIG,f"{run_id}-epoch-{epoch}")
        return evaluation["evaluate_scored_set"](score_dir,"early_stop")["metrics"]
    baseline = early_stop_evaluator(model,-1)
    print("Frozen-base pilot:",baseline)
    model_manifest = train_qwen(model,tokenizer,yes_no,ARTIFACT_DIR/"fit",
                               ARTIFACT_DIR/"model",CONFIG,early_stop_evaluator)
    print("Best adapter:",model_manifest["adapter"])
    print("Training profile:",model_manifest["training_profile"])
